In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py
from dateutil.relativedelta import relativedelta

In [ ]:
# Find developers whose personal or business address is in these states.
STATES = "'IL', 'WI', 'IN', 'MI', 'OH'"
MYSQL_DATE_FORMAT = "%Y-%m-%d"
TWO_YEARS_AGO = ((datetime.datetime.now() - relativedelta(years=2))
                .strftime(MYSQL_DATE_FORMAT))
db = Db("~/.clover/p801.cfg")
query = """
    SELECT
        d.name,
        d.approval_status,
        d.created_time,
        a.last_login,
        d.first_name,
        d.last_name,
        a.name AS "account.name",
        d.email AS "developer.email",
        a.email AS "account.email",
        d.city,
        d.state,
        d.postal_code,
        d.business_legal_name,
        d.business_city,
        d.business_state
    FROM developer d
        JOIN account a ON a.id = d.owner_account_id
    WHERE d.name NOT IN ("Clover")
    AND d.approval_status != "DENIED"
    AND (d.state in ("""+STATES+""") OR d.business_state in ("""+STATES+"""))
    AND (a.last_login > '"""+TWO_YEARS_AGO+"""' OR d.approval_status = "APPROVED")
    ORDER BY d.state, d.city, d.approval_status DESC, a.last_login DESC
    """
df = pd.read_sql(query, con=db.conn)
db.close()
df

In [ ]:
# Export results to CSV.
import string
filename = "developers_by_location_" + "-".join(STATES.translate(None, string.punctuation).split(" ")) + ".csv"
csvloc = os.environ['HOME'] + "/" + filename
print("Saving CSV to " + csvloc)
try:
    df.to_csv(csvloc)
except UnicodeEncodeError:
    df.to_csv(csvloc, encoding='utf-8')